In [113]:
import requests 
import json 
import pandas as pd 

In [115]:
url = 'https://www.reddit.com/r/food/top.json?count=25'

headers = {'User-Agent': 'Mozilla/5.0'}  

r = requests.get(url, headers=headers)
data = r.json()
data

with open('main_page.json', 'w') as main_page:
    json.dump(data, main_page)

In [117]:
next_page= data['data']['after']
if next_page: 
    next_url = f'{url}&after={next_page}'
    print(next_url) 

https://www.reddit.com/r/food/top.json?count=25&after=t3_1k92od4


In [131]:
url = 'https://www.reddit.com/r/food/top.json?count=25&after=t3_1k92od4'
headers = {'User-Agent': 'Mozilla/5.0'}

r = requests.get(url, headers=headers)
data = r.json()
data

with open('main_page_2.json', 'w') as main_page_2: 
    json.dump(data, main_page_2)

In [123]:
with open('main_page.json', 'r') as main:
    main_json = json.load(main)  

first_post = main_json['data']['children'][0]['data']

print(len(first_post))

for key, value in first_post.items():
    if isinstance(value, (list, dict)):  
        print(key + ':' + type(value).__name__)

107
link_flair_richtext:list
media_embed:dict
user_reports:list
secure_media_embed:dict
author_flair_richtext:list
gildings:dict
preview:dict
all_awardings:list
awarders:list
treatment_tags:list
mod_reports:list


In [125]:
rows = []

for post in main_json['data']['children']:
    data = post['data']
    
    row = {
        'selftext': data.get('selftext', ''),
        'author_': data.get('author', ''),
        'fullname': data.get('name', ''),
        'ups': data.get('ups', 0),
        'downs': data.get('downs', 0),
        'upvote_ratio': data.get('upvote_ratio', 0),
        'created_utc': data.get('created_utc', 0),
        'subreddit_': data.get('subreddit', ''),
        'subscribers': data.get('subreddit_subscribers', 0),
        'url': data.get('url', '')
    }
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv('main.csv', index=False)
df

,selftext,author_,fullname,ups,downs,upvote_ratio,created_utc,subreddit_,subscribers,url
0,"Best Milkshakes ever! Whitespot, Burnaby Bc",Doogsfx,t3_1k9fdv2,1701,0,0.98,1.745790e+09,food,24405510,https://i.redd.it/jk02afwt6gxe1.jpeg
1,,Notorious2again,t3_1k9k3nm,657,0,0.99,1.745804e+09,food,24405510,https://i.redd.it/v78fj47nchxe1.jpeg
2,My kids requested again. Mandolin cut russet p...,Jamieson22,t3_1k9juo6,622,0,0.99,1.745804e+09,food,24405510,https://www.reddit.com/gallery/1k9juo6
3,,EditorRedditer,t3_1k9bvmu,512,0,0.87,1.745781e+09,food,24405510,https://i.redd.it/ffycbd4rffxe1.jpeg
4,"Unexpected heatwave, so roast dinner outside, ...",Caramelotron,t3_1k996pg,500,0,0.97,1.745774e+09,food,24405510,https://www.reddit.com/gallery/1k996pg
5,,Diggy2025,t3_1k94ha9,463,0,0.99,1.745762e+09,food,24405510,https://i.redd.it/nq52w51sudxe1.jpeg
6,Love making this stuff! Recipe below:\n\n1-2 c...,No-Amphibian689,t3_1k9jm6m,409,0,0.98,1.745803e+09,food,24405510,https://i.redd.it/9syx2dm08hxe1.jpeg
7,Burger night! This week’s special had home gro...,No_Pattern3088,t3_1k99m6v,392,0,0.98,1.745775e+09,food,24405510,https://www.reddit.com/gallery/1k99m6v
8,,seescott11,t3_1k95tep,358,0,0.84,1.745766e+09,food,24405510,https://www.reddit.com/gallery/1k95tep
9,,TXwildthing99,t3_1k97ph5,333,0,0.97,1.745771e+09,food,24405510,https://i.redd.it/lsqn2i67kexe1.jpeg


In [135]:
#need to loop through original comments & find any replies to those(and replies to replies) and show all 
url = 'https://www.reddit.com/r/food/comments/1k9juo6/homemade_french_fries.json'

headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
comments_json = response.json() #set up like before

def find_comments(comment_list, rows): #need a function to go through all comments
    
    for comment in comment_list:
        data = comment.get('data', [])
        
        if comment.get('kind') == 't1': #t1 means that it is a comment and not a reply- want to show this data 
            rows.append({
                'body': data.get('body', ''),
                'author_': data.get('author', ''),
                'fullname': data.get('name', ''),
                'ups': data.get('ups', 0),
                'downs': data.get('downs', 0),
                'created_utc': data.get('created_utc', 0),
                'parent_id': data.get('parent_id', ''),
                'permalink': data.get('permalink', '')
            })
            
            replies = data.get('replies') #finds replies to original comments
            
            if isinstance(replies, dict):
                find_comments(replies['data']['children'], rows)#should show replies to og comments

rows = []
find_comments(comments_json[1]['data']['children'], rows)
df = pd.DataFrame(rows)
df.to_csv('comments.csv', index=False)
df

,body,author_,fullname,ups,downs,created_utc,parent_id,permalink
0,What deep fryer model is that? Do you recommen...,Muthafuckaaaaa,t1_mpf1ane,13,0,1.745806e+09,t3_1k9juo6,/r/food/comments/1k9juo6/homemade_french_fries...
1,https://preview.redd.it/cylqepioihxe1.jpeg?wid...,Jamieson22,t1_mpf403v,30,0,1.745807e+09,t1_mpf1ane,/r/food/comments/1k9juo6/homemade_french_fries...
2,"Damn, thanks a lot for the in-depth explanatio...",Muthafuckaaaaa,t1_mpf69jd,6,0,1.745808e+09,t1_mpf403v,/r/food/comments/1k9juo6/homemade_french_fries...
3,I don’t do it often though will say even froze...,Jamieson22,t1_mpf6x5t,3,0,1.745808e+09,t1_mpf69jd,/r/food/comments/1k9juo6/homemade_french_fries...
4,Appreciate you 🙏🏼,Muthafuckaaaaa,t1_mpf8m0v,2,0,1.745809e+09,t1_mpf6x5t,/r/food/comments/1k9juo6/homemade_french_fries...
5,Thanks u/Muthafuckaaaa!,Jamieson22,t1_mpfasyh,3,0,1.745810e+09,t1_mpf8m0v,/r/food/comments/1k9juo6/homemade_french_fries...
6,\+1 recommendation for this fryer - I have thi...,Lazlow_Panaflex,t1_mpgyww1,2,0,1.745843e+09,t1_mpf69jd,/r/food/comments/1k9juo6/homemade_french_fries...
7,&gt;scary for knuckles\n\nNo kidding - I have ...,royals796,t1_mpg6r0x,1,0,1.745827e+09,t1_mpf403v,/r/food/comments/1k9juo6/homemade_french_fries...
8,"Looks really good, but gonna need to fill up t...",americanmuscle1988,t1_mpevznz,10,0,1.745804e+09,t3_1k9juo6,/r/food/comments/1k9juo6/homemade_french_fries...
9,I have been craving these kind of fries for a ...,NeuHundred,t1_mpeytrd,4,0,1.745805e+09,t3_1k9juo6,/r/food/comments/1k9juo6/homemade_french_fries...
